In [290]:
import polars as pl
import _internal as plc
import polars.selectors as cs
import numpy as np
from scipy.stats import chi2_contingency, fisher_exact

In [209]:
bnc = pl.read_parquet("bnc.parquet")
speakers = pl.read_parquet("bnc-speakers.parquet")

In [210]:
speakers = speakers.filter(pl.col("sex").is_in(["m", "f"]))
speakers.group_by("sex").len()

sex,len
str,u32
"""m""",2453
"""f""",1368


In [211]:
bnc = (
    bnc.filter(pl.col("text_type") == "CONVRSN")
    .join(speakers, on="speaker_id", how="inner")
    .with_columns(pl.col("token").str.to_lowercase().alias("norm"))
)

In [212]:
table = (
    bnc.group_by((pl.col("norm") == "fox").alias("is_fuck"), "sex")
    .agg(pl.col("norm").len().alias("count"))
    .with_columns(rel_freq=pl.col("count") / pl.col("count").sum().over("sex"))
    .sort(by=["sex", "is_fuck"])
)
table

is_fuck,sex,count,rel_freq
bool,str,u32,f64
false,"""f""",2745522,0.999993
true,"""f""",20,0.000007
false,"""m""",1784710,0.999993
true,"""m""",13,0.000007


In [213]:
table = np.array(table["count"]).reshape(2, 2)
table

array([[2745522,      20],
       [1784710,      13]], dtype=uint32)

In [214]:
fisher_exact(table)

SignificanceResult(statistic=np.float64(0.9999323699648682), pvalue=1.0)

In [215]:
chi2_contingency(table)

Chi2ContingencyResult(statistic=np.float64(0.0), pvalue=np.float64(1.0), dof=1, expected_freq=array([[2.74552200e+06, 1.99994671e+01],
       [1.78471000e+06, 1.30005329e+01]]))

In [216]:
np.sqrt(chi2_contingency(table).statistic / table.sum())

np.float64(0.0)

In [217]:
plc.compute_loglik(plc.crosstab(bnc, pl.col("norm").str.to_lowercase(), "sex")).filter(
    pl.col("sex") == "m", pl.col("f12") > pl.col("f1") * (pl.col("f2") / pl.col("n"))
).sort(by="loglik", descending=True)

norm,sex,f12,f1,f2,n,loglik
str,str,u32,u32,u32,u32,f64
"""fucking""","""m""",1383,1709,1784723,4530265,1237.920581
"""er""","""m""",9415,18752,1784723,4530265,900.794048
"""the""","""m""",43385,100752,1784723,4530265,574.376875
""",""","""m""",84406,203556,1784723,4530265,380.46747
"""yeah""","""m""",21888,50681,1784723,4530265,305.64689
…,…,…,…,…,…,…
"""flavour""","""m""",15,38,1784723,4530265,0.000097
"""pe""","""m""",15,38,1784723,4530265,0.000097
"""charges""","""m""",15,38,1784723,4530265,0.000097


In [218]:
plc.compute_loglik(plc.crosstab(bnc, pl.col("norm").str.to_lowercase(), "sex")).filter(
    pl.col("f12") > pl.col("f1") * (pl.col("f2") / pl.col("n"))
).sort(by="loglik", descending=True)

norm,sex,f12,f1,f2,n,loglik
str,str,u32,u32,u32,u32,f64
"""she""","""f""",22807,29844,2745542,4530265,3373.99179
"""fucking""","""m""",1383,1709,1784723,4530265,1237.920581
"""her""","""f""",7306,9619,2745542,4530265,1017.03576
"""said""","""f""",12375,17286,2745542,4530265,915.450508
"""er""","""m""",9415,18752,1784723,4530265,900.794048
…,…,…,…,…,…,…
"""shove""","""f""",40,66,2745542,4530265,7.1989e-8
"""fox""","""f""",20,33,2745542,4530265,3.5384e-8
"""trips""","""f""",20,33,2745542,4530265,3.5384e-8


In [219]:
plc.compute_loglik(
    plc.crosstab(bnc, pl.col("norm").str.to_lowercase(), "ageGroup")
).filter(pl.col("f12") > pl.col("f1") * (pl.col("f2") / pl.col("n"))).sort(
    by="loglik", descending=True
).filter(pl.col("ageGroup") == "Ag0")

norm,ageGroup,f12,f1,f2,n,loglik
str,str,u32,u32,u32,u32,f64
"""mum""","""Ag0""",1616,3500,448945,4530265,3036.933686
""".""","""Ag0""",31629,249981,448945,4530265,2077.74327
"""mummy""","""Ag0""",808,1497,448945,4530265,1814.674332
"""<unclear/>""","""Ag0""",10401,72369,448945,4530265,1463.654411
"""!""","""Ag0""",7253,48947,448945,4530265,1178.924243
…,…,…,…,…,…,…
"""meters""","""Ag0""",1,10,448945,4530265,0.000091
"""worlds""","""Ag0""",1,10,448945,4530265,0.000091
"""saw""","""Ag0""",109,1099,448945,4530265,0.000083


In [252]:
k = 100
ctab = (
    plc.crosstab(bnc, pl.col("norm").str.to_lowercase(), "sex")
    .select("norm", "sex", (pl.col("f12") / pl.col("f2") * 1000000).alias("rel_freq"))
    .pivot(on="sex", index="norm")
    .drop_nulls()
    .with_columns(smp=(pl.col("f") + k) / (pl.col("m") + k))
    .sort(by="smp", descending=True)
)

ctab

norm,m,f,smp
str,f64,f64,f64
"""she""",3942.908788,8306.920819,2.079424
"""her""",1295.999435,2661.041062,1.977824
"""charlotte""",13.447465,108.539589,1.838204
"""christmas""",159.688646,366.047942,1.794641
"""lovely""",227.486282,443.264026,1.658891
…,…,…,…
"""jesus""",99.175054,13.112165,0.567903
"""aye""",652.20205,319.062684,0.557115
"""fuck""",185.462954,38.608042,0.485555


In [313]:
# https://inquirer.sites.fas.harvard.edu/spreadsheet_guide.htm
# merge rows for multi-sense words?

inq = (
    pl.read_csv("inqtabs.tsv", separator="\t")
    .with_columns(pl.col("Entry").str.to_lowercase())
    .tail(-1)
    .select(cs.all().exclude("Source"))
    .with_columns(~cs.all().exclude("Entry").is_null())
)
inq

Entry,Positiv,Negativ,Pstv,Affil,Ngtv,Hostile,Strong,Power,Weak,Submit,Active,Passive,Pleasur,Pain,Feel,Arousal,EMOT,Virtue,Vice,Ovrst,Undrst,Academ,Doctrin,Econ@,Exch,ECON,Exprsv,Legal,Milit,Polit@,POLIT,Relig,Role,COLL,Work,Ritual,…,WltTran,WltOth,WltTot,WlbGain,WlbLoss,WlbPhys,WlbPsyc,WlbPt,WlbTot,EnlGain,EnlLoss,EnlEnds,EnlPt,EnlOth,EnlTot,SklAsth,SklPt,SklOth,SklTot,TrnGain,TrnLoss,TranLw,MeansLw,EndsLw,ArenaLw,PtLw,Nation,Anomie,NegAff,PosAff,SureLw,If,NotLw,TimeSpc,FormLw,Othtags,Defined
str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,…,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
"""abandon""",false,true,false,false,true,false,false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,…,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true,true
"""abandonment""",false,true,false,false,false,false,false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,…,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true,true
"""abate""",false,true,false,false,false,false,false,false,false,false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,…,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,true,true
"""abatement""",false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,…,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true,false
"""abdicate""",false,true,false,false,false,false,false,false,true,true,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,…,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""zenith""",true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,…,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true,true
"""zero""",false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,…,false,false,false,false,false,false,false,fals

In [319]:
ctab.join(inq, left_on="norm", right_on="Entry", how="left").filter("Kin@").group_by(
    pl.col("smp") >= 1
).len()

smp,len
bool,u32
false,13
true,21


In [320]:
ctab.join(inq, left_on="norm", right_on="Entry", how="left").filter("Strong").group_by(
    pl.col("smp") >= 1
).len()

smp,len
bool,u32
true,140
false,308


In [321]:
ctab.join(inq, left_on="norm", right_on="Entry", how="left").filter("Weak").group_by(
    pl.col("smp") >= 1
).len()

smp,len
bool,u32
true,90
false,108


In [322]:
ctab.join(inq, left_on="norm", right_on="Entry", how="left").filter("Hostile").group_by(
    pl.col("smp") >= 1
).len()

smp,len
bool,u32
true,51
false,113


In [311]:
inq.group_by("Kin@").len()

Kin@,len
str,u32
"""Kin@""",50
null,11738
